# Import and Model Loading:

In [1]:
import sys
from pathlib import Path

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to Python path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from xray.interpretability import *

c:\Users\megha\Desktop\xray-transformers\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#IMPORTS:

import torch
import matplotlib.pyplot as plt

from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizer
)

from xray.interpretability import DistilBertInterpreter

# LOAD THE HUGGINGFACE MODEL:

# Use GPU if available
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

model = DistilBertForSequenceClassification.from_pretrained(
    "JDsouza1/distilbert-imdb-sentiment"
).to(device)

tokenizer = DistilBertTokenizer.from_pretrained(
    "JDsouza1/distilbert-imdb-sentiment"
)

print("Model loaded!")

Device: cpu


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 11666.42it/s]


Model loaded!


In [12]:
# CREATE THE INTERPRETER:

interpreter = DistilBertInterpreter(
    model,
    tokenizer,
    device
)

# Test Cases:

In [4]:
test_texts = [
    "This movie was absolutely fantastic and I loved every minute of it.",
    "This movie was terrible, boring, and completely disappointing.",
    "This was not a bad movie at all.",
    "The acting was excellent, but the story was painfully boring.",
    "I expected this movie to be terrible, but it was actually wonderful.",
    "What a fantastic movie... if you enjoy falling asleep."
]

for i, text in enumerate(test_texts, 1):
    print(f"{i}. {text}")

1. This movie was absolutely fantastic and I loved every minute of it.
2. This movie was terrible, boring, and completely disappointing.
3. This was not a bad movie at all.
4. The acting was excellent, but the story was painfully boring.
5. I expected this movie to be terrible, but it was actually wonderful.
6. What a fantastic movie... if you enjoy falling asleep.


# Original Predictions:

In [5]:
def predict_text(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=-1)

    predicted_class = torch.argmax(probabilities, dim=-1).item()
    confidence = probabilities[0, predicted_class].item()

    return predicted_class, confidence

In [6]:
original_results = []

for text in test_texts:
    prediction, confidence = predict_text(text)

    original_results.append({
        "text": text,
        "prediction": prediction,
        "label": "POSITIVE" if prediction == 1 else "NEGATIVE",
        "confidence": confidence
    })

original_df = pd.DataFrame(original_results)

original_df

,text,prediction,label,confidence
0,This movie was absolutely fantastic and I love...,1,POSITIVE,0.995928
1,"This movie was terrible, boring, and completel...",0,NEGATIVE,0.997209
2,This was not a bad movie at all.,1,POSITIVE,0.825081
3,"The acting was excellent, but the story was pa...",0,NEGATIVE,0.978495
4,"I expected this movie to be terrible, but it w...",1,POSITIVE,0.986044
5,What a fantastic movie... if you enjoy falling...,1,POSITIVE,0.989481


# Word Removal Perturbation:

In [8]:
text = test_texts[0]

words = text.split()

original_prediction, original_confidence = predict_text(text)

perturbation_results = []

for i, word in enumerate(words):
    perturbed_words = words[:i] + words[i+1:]
    perturbed_text = " ".join(perturbed_words)

    prediction, confidence = predict_text(perturbed_text)

    perturbation_results.append({
        "removed_word": word,
        "perturbed_text": perturbed_text,
        "prediction": prediction,
        "confidence": confidence,
        "confidence_change": confidence - original_confidence
    })

perturbation_df = pd.DataFrame(perturbation_results)

perturbation_df

,removed_word,perturbed_text,prediction,confidence,confidence_change
0,This,movie was absolutely fantastic and I loved eve...,1,0.995387,-0.000540
1,movie,This was absolutely fantastic and I loved ever...,1,0.995328,-0.000599
2,was,This movie absolutely fantastic and I loved ev...,1,0.996060,0.000132
3,absolutely,This movie was fantastic and I loved every min...,1,0.996027,0.000100
4,fantastic,This movie was absolutely and I loved every mi...,1,0.995801,-0.000127
5,and,This movie was absolutely fantastic I loved ev...,1,0.995668,-0.000259
6,I,This movie was absolutely fantastic and loved ...,1,0.995809,-0.000118
7,loved,This movie was absolutely fantastic and I ever...,1,0.995014,-0.000914
8,every,This movie was absolutely fantastic and I love...,1,0.995696,-0.000231
9,minute,This movie was absolutely fantastic and I love...,1,0.996092,0.000164


# Compare attribution with perturbation effect:

In [10]:
def get_word_attributions(text):
    """
    Generate word-level Integrated Gradients
    for a given input sentence.
    """

    (
        tokens,
        attributions,
        delta,
        input_output,
        baseline_output,
        total_attribution,
        completeness_error
    ) = interpreter.attribute(text)

    words, scores = interpreter.aggregate_tokens(
        tokens,
        attributions
    )

    return words, scores

In [13]:
words, attributions = get_word_attributions(text)

attribution_df = pd.DataFrame({
    "word": words,
    "attribution": [float(a) for a in attributions]
})

attribution_df

,word,attribution
0,this,0.217276
1,movie,0.148403
2,was,0.045538
3,absolutely,0.234302
4,fantastic,0.472755
5,and,0.427234
6,i,0.217258
7,loved,0.485604
8,every,0.038579
9,minute,0.014611
